In [7]:
from pathlib import Path
from urllib.request import urlretrieve
from zipfile import ZipFile
import json

import geopandas as gpd
import pandas as pd
from shapely.geometry import LineString, MultiLineString, MultiPoint, Point
from shapely.geometry.base import BaseGeometry
from shapely.ops import unary_union

# =========================
# Configuration
# =========================
BASE_DIR = Path.cwd()
CACHE_DIR = BASE_DIR / "_cache_border_controls"
D_ROOT = Path("D:/border_points_work")
ROADS_ZIP_DIR = D_ROOT / "roads_zip"
ROADS_DB_DIR = D_ROOT / "roads_db"
OUTPUT_GPKG = Path.home() / "Desktop" / "border_point.gpkg"
POINTS_LAYER = "border_control_points"
SAVE_SHARED_BORDERS = True

NATURAL_EARTH_URL = (
    "https://naturalearth.s3.amazonaws.com/10m_cultural/"
    "ne_10m_admin_0_countries.zip"
)
GEOFABRIK_INDEX_URL = "https://download.geofabrik.de/index-v1.json"
GEOBOUNDARIES_API_COUNTRY_URL = "https://www.geoboundaries.org/api/current/gbOpen/{iso3}/ADM0/"
USE_PRECISE_BORDERS = True
AFRICA_ROADS_GPKG = Path(r"D:/geo data/ATSD road+rail+air/africa_roads_network.gpkg")

# Main roads only.
MAJOR_HIGHWAYS = {
    "motorway",
    "motorway_link",
    "trunk",
    "trunk_link",
    "primary",
    "primary_link",
}

# Distance along road used to verify a true crossing (about 5-6 m at equator).
CROSSING_CHECK_STEP_DEG = 0.00005
# Buffer used for side classification to absorb minor border misalignments.
CROSSING_SIDE_BUFFER_DEG = 0.00005
# Tolerance used when edge and border lines are nearly touching but not exactly intersecting.
BORDER_INTERSECTION_TOLERANCE_DEG = 0.0001


def ensure_parent(path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)


def ensure_dir(path: Path) -> None:
    path.mkdir(parents=True, exist_ok=True)


def download_if_missing(url: str, target_path: Path, label: str) -> Path:
    ensure_parent(target_path)
    if target_path.exists():
        print(f"[cache] {label}: {target_path}")
        return target_path
    print(f"[download] {label}: {url}")
    urlretrieve(url, target_path)
    return target_path


def try_download_if_missing(url: str, target_path: Path, label: str) -> Path | None:
    try:
        return download_if_missing(url, target_path, label)
    except Exception as exc:
        print(f"[warn] Could not download {label}: {exc}")
        return None


def assert_d_drive_ready() -> None:
    if not Path("D:/").exists():
        raise RuntimeError(
            "Drive D: not found. Please create/select drive D or change D_ROOT in the configuration."
        )
    ensure_dir(ROADS_ZIP_DIR)
    ensure_dir(ROADS_DB_DIR)


def extract_lines(geom: BaseGeometry) -> list[LineString]:
    if geom.is_empty:
        return []
    if isinstance(geom, LineString):
        return [geom]
    if isinstance(geom, MultiLineString):
        return [g for g in geom.geoms if not g.is_empty]
    if hasattr(geom, "geoms"):
        out = []
        for part in geom.geoms:
            out.extend(extract_lines(part))
        return out
    return []


def extract_points(geom: BaseGeometry) -> list[Point]:
    if geom.is_empty:
        return []
    if isinstance(geom, Point):
        return [geom]
    if isinstance(geom, MultiPoint):
        return [g for g in geom.geoms if not g.is_empty]
    if hasattr(geom, "geoms"):
        out = []
        for part in geom.geoms:
            out.extend(extract_points(part))
        return out
    return []


def _first_existing_column(columns: list[str], candidates: list[str]) -> str | None:
    lower_map = {c.lower(): c for c in columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None


def load_precise_world_boundaries(path: Path | None) -> gpd.GeoDataFrame | None:
    if path is None or not path.exists():
        return None
    try:
        world = gpd.read_file(path)
        if world.empty:
            return None
        if world.crs is None:
            world = world.set_crs(4326, allow_override=True)
        else:
            world = world.to_crs(4326)
        world = world[world.geometry.notnull() & ~world.geometry.is_empty].copy()
        print(f"[info] Loaded precise boundaries source: {path}")
        return world
    except Exception as exc:
        print(f"[warn] Could not read precise boundaries file: {exc}")
        return None


def load_precise_boundaries_for_countries(countries: gpd.GeoDataFrame) -> gpd.GeoDataFrame | None:
    cache_dir = CACHE_DIR / "boundaries" / "geoBoundaries"
    ensure_dir(cache_dir)

    iso3_values = sorted(
        {
            str(v).strip().upper()
            for v in countries["iso_a3"].dropna().tolist()
            if str(v).strip() and str(v).strip().upper() != "-99" and len(str(v).strip()) == 3
        }
    )

    if not iso3_values:
        print("[warn] No ISO3 codes available for precise boundaries download.")
        return None

    geoms = []
    ok = 0

    for iso3 in iso3_values:
        meta_url = GEOBOUNDARIES_API_COUNTRY_URL.format(iso3=iso3)
        meta_path = try_download_if_missing(
            meta_url,
            cache_dir / f"{iso3}_ADM0_meta.json",
            label=f"geoBoundaries metadata {iso3}",
        )
        if meta_path is None:
            continue

        try:
            with open(meta_path, "r", encoding="utf-8") as f:
                meta = json.load(f)
        except Exception as exc:
            print(f"[warn] Could not parse geoBoundaries metadata for {iso3}: {exc}")
            continue

        gj_url = meta.get("gjDownloadURL") or meta.get("simplifiedGeometryGeoJSON")
        if not gj_url:
            print(f"[warn] No GeoJSON URL found in geoBoundaries metadata for {iso3}.")
            continue

        gj_path = try_download_if_missing(
            gj_url,
            cache_dir / f"{iso3}_ADM0.geojson",
            label=f"geoBoundaries ADM0 {iso3}",
        )
        if gj_path is None:
            continue

        gdf = load_precise_world_boundaries(gj_path)
        if gdf is None or gdf.empty:
            continue

        gdf = gdf[["geometry"]].copy()
        gdf["shapeGroup"] = iso3
        geoms.append(gdf)
        ok += 1

    if not geoms:
        print("[warn] Could not load any precise country boundaries from geoBoundaries.")
        return None

    precise_world = gpd.GeoDataFrame(pd.concat(geoms, ignore_index=True), geometry="geometry", crs=4326)
    print(f"[info] Loaded precise boundaries for {ok}/{len(iso3_values)} countries")
    return precise_world


def apply_precise_boundaries(
    countries: gpd.GeoDataFrame,
    precise_world: gpd.GeoDataFrame | None,
) -> gpd.GeoDataFrame:
    if precise_world is None or precise_world.empty:
        return countries

    cols = list(precise_world.columns)
    iso3_col = _first_existing_column(cols, ["shapeGroup", "shapeISO", "ISO_A3", "iso3", "adm0_a3"])
    iso2_col = _first_existing_column(cols, ["shapeISO", "ISO_A2", "iso2", "adm0_a2"])

    map_iso3: dict[str, BaseGeometry] = {}
    map_iso2: dict[str, BaseGeometry] = {}

    if iso3_col is not None:
        for _, row in precise_world[[iso3_col, "geometry"]].dropna().iterrows():
            code = str(row[iso3_col]).strip().upper()
            if len(code) == 3:
                map_iso3[code] = row.geometry

    if iso2_col is not None:
        for _, row in precise_world[[iso2_col, "geometry"]].dropna().iterrows():
            code = str(row[iso2_col]).strip().upper()
            if len(code) == 2:
                map_iso2[code] = row.geometry

    if not map_iso3 and not map_iso2:
        print("[warn] Precise boundaries loaded but no ISO columns were detected.")
        return countries

    out = countries.copy()
    new_geoms = []
    replaced = 0

    for _, row in out.iterrows():
        iso3 = str(row["iso_a3"]).upper() if pd.notna(row["iso_a3"]) else ""
        iso2 = str(row["iso_a2"]).upper() if pd.notna(row["iso_a2"]) else ""

        geom = None
        if iso3 in map_iso3:
            geom = map_iso3[iso3]
        elif iso2 in map_iso2:
            geom = map_iso2[iso2]

        if geom is not None:
            new_geoms.append(geom)
            replaced += 1
        else:
            new_geoms.append(row.geometry)

    out["geometry"] = new_geoms
    out = out[out.geometry.notnull() & ~out.geometry.is_empty].copy()
    out = out.set_crs(4326, allow_override=True)
    print(f"[info] Precise boundaries matched for {replaced}/{len(out)} countries")
    return out


def get_subsaharan_countries(
    boundaries_zip: Path,
    precise_world: gpd.GeoDataFrame | None = None,
) -> gpd.GeoDataFrame:
    countries = gpd.read_file(boundaries_zip)
    countries = countries[
        (countries["CONTINENT"] == "Africa")
        & countries["SUBREGION"].isin(
            ["Eastern Africa", "Middle Africa", "Southern Africa", "Western Africa"]
        )
    ][["NAME", "ISO_A2", "ISO_A3", "geometry"]].copy()

    countries.rename(
        columns={"NAME": "country", "ISO_A2": "iso_a2", "ISO_A3": "iso_a3"},
        inplace=True,
    )
    countries = countries[countries.geometry.notnull() & ~countries.geometry.is_empty].copy()
    countries = countries.set_crs(4326, allow_override=True).reset_index(drop=True)

    countries = apply_precise_boundaries(countries, precise_world)
    countries.reset_index(drop=True, inplace=True)
    return countries


def build_shared_borders(countries: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    countries = countries.reset_index(drop=True).copy()
    sindex = countries.sindex
    pairs: dict[tuple[str, str], dict[str, object]] = {}

    print("[compute] Building shared international borders")
    for idx, row in countries.iterrows():
        for other_idx in sindex.query(row.geometry, predicate="touches"):
            other_idx = int(other_idx)
            if other_idx <= idx:
                continue

            other = countries.iloc[other_idx]
            lines = extract_lines(row.geometry.boundary.intersection(other.geometry.boundary))
            if not lines:
                continue

            key = tuple(sorted((str(row["iso_a3"]).upper(), str(other["iso_a3"]).upper())))
            if key not in pairs:
                if str(row["iso_a3"]).upper() <= str(other["iso_a3"]).upper():
                    c1, c2 = row["country"], other["country"]
                else:
                    c1, c2 = other["country"], row["country"]
                pairs[key] = {"country1": c1, "country2": c2, "lines": []}
            pairs[key]["lines"].extend(lines)

    records = [
        {
            "iso3_1": iso3_1,
            "iso3_2": iso3_2,
            "country1": data["country1"],
            "country2": data["country2"],
            "geometry": unary_union(data["lines"]),
        }
        for (iso3_1, iso3_2), data in pairs.items()
        if data["lines"]
    ]
    if not records:
        print("[warn] No shared borders were found from current country geometries.")
        return gpd.GeoDataFrame(columns=["iso3_1", "iso3_2", "country1", "country2", "geometry"], geometry="geometry", crs=4326)
    border_gdf = gpd.GeoDataFrame(records, geometry="geometry", crs=4326)
    border_gdf = border_gdf[border_gdf.geometry.notnull() & ~border_gdf.geometry.is_empty].copy()
    print(f"[compute] Shared borders found: {len(border_gdf)}")
    return border_gdf


def load_geofabrik_index(index_json_path: Path) -> dict:
    with open(index_json_path, "r", encoding="utf-8") as f:
        return json.load(f)


def geofabrik_iso2_to_shp(index_data: dict) -> dict[str, str]:
    mapping: dict[str, str] = {}
    for feat in index_data.get("features", []):
        props = feat.get("properties", {})
        shp_url = props.get("urls", {}).get("shp")
        iso_list = props.get("iso3166-1:alpha2", [])
        if not shp_url or not iso_list:
            continue
        for iso2 in iso_list:
            if isinstance(iso2, str) and len(iso2) == 2:
                mapping[iso2.upper()] = shp_url
    return mapping


def find_roads_shp_in_zip(zip_path: Path) -> str | None:
    with ZipFile(zip_path, "r") as zf:
        for name in zf.namelist():
            if name.lower().endswith("roads_free_1.shp"):
                return name
    return None


def _iso3_pair(a: str, b: str) -> tuple[str, str]:
    return tuple(sorted((a, b)))


def build_border_control_points_from_africa_edges(
    shared_borders: gpd.GeoDataFrame,
    countries: gpd.GeoDataFrame,
    roads_gpkg_path: Path,
) -> gpd.GeoDataFrame:
    if not roads_gpkg_path.exists():
        raise FileNotFoundError(f"Road network not found: {roads_gpkg_path}")

    print(f"[compute] Loading Africa-wide roads edges: {roads_gpkg_path}")
    edges = gpd.read_file(roads_gpkg_path, layer="edges")

    required_cols = {"from_iso3", "to_iso3", "geometry"}
    missing = required_cols.difference(set(edges.columns))
    if missing:
        raise KeyError(f"Missing required columns in edges layer: {sorted(missing)}")

    edges = edges[edges.geometry.notnull() & ~edges.geometry.is_empty].copy()
    edges["from_iso3"] = edges["from_iso3"].astype(str).str.upper().str.strip()
    edges["to_iso3"] = edges["to_iso3"].astype(str).str.upper().str.strip()

    # Keep only explicit cross-border edges.
    edges = edges[
        (edges["from_iso3"].str.len() == 3)
        & (edges["to_iso3"].str.len() == 3)
        & (edges["from_iso3"] != edges["to_iso3"])
    ].copy()

    ssa_iso3 = set(countries["iso_a3"].dropna().astype(str).str.upper())
    edges = edges[edges["from_iso3"].isin(ssa_iso3) & edges["to_iso3"].isin(ssa_iso3)].copy()
    print(f"[compute] Cross-border edges in Sub-Saharan set: {len(edges)}")

    if edges.empty:
        return gpd.GeoDataFrame(columns=["iso3_1", "iso3_2", "country1", "country2", "lon", "lat", "geometry"], geometry="geometry", crs=4326)

    shared = shared_borders.copy()
    shared["pair_key"] = shared.apply(lambda r: _iso3_pair(r["iso3_1"], r["iso3_2"]), axis=1)
    pair_to_border = {row["pair_key"]: row.geometry for _, row in shared.iterrows()}

    edges["pair_key"] = edges.apply(lambda r: _iso3_pair(r["from_iso3"], r["to_iso3"]), axis=1)
    edges = edges[edges["pair_key"].isin(pair_to_border)].copy()
    print(f"[compute] Cross-border edges matching a known shared border: {len(edges)}")

    if edges.empty:
        return gpd.GeoDataFrame(columns=["iso3_1", "iso3_2", "country1", "country2", "lon", "lat", "geometry"], geometry="geometry", crs=4326)

    pair_to_names = {
        _iso3_pair(row["iso3_1"], row["iso3_2"]): (row["country1"], row["country2"])
        for _, row in shared_borders.iterrows()
    }

    records = []
    for _, edge_row in edges.iterrows():
        pair = edge_row["pair_key"]
        border_geom = pair_to_border.get(pair)
        if border_geom is None:
            continue

        inter = edge_row.geometry.intersection(border_geom)
        points = extract_points(inter)
        if not points:
            inter_tol = edge_row.geometry.intersection(border_geom.buffer(BORDER_INTERSECTION_TOLERANCE_DEG))
            points = extract_points(inter_tol)
        if not points:
            continue

        country1, country2 = pair_to_names.get(pair, (pair[0], pair[1]))
        for pt in points:
            rec = {
                "iso3_1": pair[0],
                "iso3_2": pair[1],
                "country1": country1,
                "country2": country2,
                "lon": round(pt.x, 6),
                "lat": round(pt.y, 6),
                "geometry": pt,
            }
            if "id" in edges.columns and pd.notna(edge_row.get("id")):
                rec["edge_id"] = edge_row.get("id")
            if "name" in edges.columns and pd.notna(edge_row.get("name")):
                rec["edge_name"] = edge_row.get("name")
            records.append(rec)

    points = gpd.GeoDataFrame(records, geometry="geometry", crs=4326)
    if points.empty:
        return points

    points = points.drop_duplicates(subset=["iso3_1", "iso3_2", "lon", "lat"]).reset_index(drop=True)
    print(f"[compute] Border crossing points found: {len(points)}")
    return points


# =========================
# Run pipeline
# =========================
assert_d_drive_ready()

boundaries_zip = download_if_missing(
    NATURAL_EARTH_URL,
    CACHE_DIR / "boundaries" / "ne_10m_admin_0_countries.zip",
    label="Natural Earth countries",
)
countries_base = get_subsaharan_countries(boundaries_zip, precise_world=None)
countries = countries_base.copy()
precise_world = None
if USE_PRECISE_BORDERS:
    precise_world = load_precise_boundaries_for_countries(countries_base)
    countries = apply_precise_boundaries(countries_base, precise_world)
    print("[info] Using Natural Earth shared borders + precise country polygons for crossing validation")
print(f"[info] Sub-Saharan countries loaded: {len(countries)}")

shared_borders = build_shared_borders(countries_base)
if shared_borders.empty:
    raise RuntimeError("No shared borders found. Check country boundaries input.")

points = build_border_control_points_from_africa_edges(
    shared_borders=shared_borders,
    countries=countries,
    roads_gpkg_path=AFRICA_ROADS_GPKG,
)

ensure_parent(OUTPUT_GPKG)
points.to_file(OUTPUT_GPKG, layer=POINTS_LAYER, driver="GPKG")
print(f"[save] Points layer written to: {OUTPUT_GPKG} (layer={POINTS_LAYER})")

if SAVE_SHARED_BORDERS:
    shared_borders.to_file(OUTPUT_GPKG, layer="shared_borders", driver="GPKG", mode="a")
    print("[save] Shared borders written to layer: shared_borders")

if not points.empty:
    points.head()

[cache] Natural Earth countries: c:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\_cache_border_controls\boundaries\ne_10m_admin_0_countries.zip
[cache] geoBoundaries metadata AGO: c:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\_cache_border_controls\boundaries\geoBoundaries\AGO_ADM0_meta.json
[cache] geoBoundaries ADM0 AGO: c:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\_cache_border_controls\boundaries\geoBoundaries\AGO_ADM0.geojson
[info] Loaded precise boundaries source: c:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\_cache_border_controls\boundaries\geoBoundaries\AGO_ADM0.geojson
[cache] geoBoundaries metadata BDI: c:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\_cache_border_controls\boundaries\geoBoundaries\BDI_ADM0_meta.json
[cache] geoBoundaries ADM0 BDI: c:\Users\Fra\Documents\OnStoveThesis\thesis\script_nostri\_cache_border_controls\boundaries\geoBoundaries\BDI_ADM0.geojson
[info] Loaded precise boundaries source: c:\Users\

In [8]:
from pathlib import Path
import geopandas as gpd
from shapely.geometry import LineString, MultiLineString, Point

# =========================
# Method 2: border_road midpoint export
# =========================

roads_gpkg_path = (
    AFRICA_ROADS_GPKG
    if "AFRICA_ROADS_GPKG" in globals()
    else Path(r"D:/geo data/ATSD road+rail+air/africa_roads_network.gpkg")
)
output_method2 = Path.home() / "Desktop" / "border_point_method2.gpkg"
output_layer = "border_points_method2"

if not roads_gpkg_path.exists():
    raise FileNotFoundError(f"Road network not found: {roads_gpkg_path}")

edges = gpd.read_file(roads_gpkg_path, layer="edges")
print(f"[info] Loaded edges: {len(edges)}")

if "border_road" not in edges.columns:
    raise KeyError("Field 'border_road' not found in edges layer")

def _to_bool(v) -> bool:
    if isinstance(v, bool):
        return v
    if v is None:
        return False
    s = str(v).strip().lower()
    return s in {"1", "true", "t", "yes", "y"}

edges = edges[edges["border_road"].apply(_to_bool)].copy()
edges = edges[edges.geometry.notnull() & ~edges.geometry.is_empty].copy()
print(f"[info] Edges with border_road=true: {len(edges)}")

def midpoint_of_edge(geom) -> Point | None:
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, LineString):
        return geom.interpolate(0.5, normalized=True)
    if isinstance(geom, MultiLineString):
        parts = [g for g in geom.geoms if not g.is_empty and isinstance(g, LineString)]
        if not parts:
            return None
        longest = max(parts, key=lambda g: g.length)
        return longest.interpolate(0.5, normalized=True)
    return None

edges["geometry"] = edges.geometry.apply(midpoint_of_edge)
points_method2 = gpd.GeoDataFrame(edges, geometry="geometry", crs=edges.crs)
points_method2 = points_method2[points_method2.geometry.notnull() & ~points_method2.geometry.is_empty].copy()
points_method2 = points_method2.to_crs(4326) if points_method2.crs is not None else points_method2.set_crs(4326, allow_override=True)
points_method2["lon"] = points_method2.geometry.x.round(6)
points_method2["lat"] = points_method2.geometry.y.round(6)

if output_method2.exists():
    output_method2.unlink()

points_method2.to_file(output_method2, layer=output_layer, driver="GPKG")
print(f"[save] Method2 points written to: {output_method2} (layer={output_layer})")
print(f"[save] Method2 point count: {len(points_method2)}")

points_method2.head()

[info] Loaded edges: 2246823
[info] Edges with border_road=true: 828
[save] Method2 points written to: C:\Users\Fra\Desktop\border_point_method2.gpkg (layer=border_points_method2)
[save] Method2 point count: 828


,id,from_id,to_id,from_iso3,to_iso3,component,border_road,corridor_name,length_m,osm_way_id,...,tag_lanes,bridge,paved,material,lanes,asset_type,source,geometry,lon,lat
3207,africa-latest_13_1015970,africa-latest_13_210941,africa-latest_13_16614,GIN,SLE,0.0,1,NaN,3629.126118,679473263.0,...,NaN,False,False,gravel,1.0,road_secondary,['Open Street Maps. Open Street Maps. https://...,POINT (-10.71564 8.32139),-10.715644,8.321385
3224,africa-latest_13_1016000,africa-latest_13_164681,africa-latest_13_164680,SLE,GIN,0.0,1,NaN,1425.274911,679499245.0,...,NaN,False,False,gravel,1.0,road_secondary,['Open Street Maps. Open Street Maps. https://...,POINT (-10.70519 8.26873),-10.705186,8.268733
3797,africa-latest_13_1021802,africa-latest_13_628921,africa-latest_13_853914,MLI,CIV,0.0,1,NaN,2853.753708,681856442.0,...,2.0,False,True,asphalt,2.0,road_trunk,['Open Street Maps. Open Street Maps. https://...,POINT (-5.6425 10.45838),-5.642499,10.458380
5762,africa-latest_13_1031918,africa-latest_13_861491,africa-latest_13_1374,GIN,MLI,0.0,1,NaN,1379.463669,685937987.0,...,NaN,False,False,ground,1.0,road_secondary,['Open Street Maps. Open Street Maps. https://...,POINT (-8.28756 11.00234),-8.287555,11.002344
6647,africa-latest_13_1036120,africa-latest_13_648499,africa-latest_13_236032,GIN,LBR,0.0,1,NaN,4184.150140,688118157.0,...,2.0,False,False,compacted,2.0,road_secondary,['Open Street Maps. Open Street Maps. https://...,POINT (-9.66256 8.48372),-9.662562,8.483719


In [ ]:
from pathlib import Path
import geopandas as gpd
import pandas as pd
import osmnx as ox
from shapely.geometry import Point, MultiPoint

# =========================
# Method 3: OSMnx direct extraction
# =========================

if "shared_borders" not in globals() or shared_borders is None or shared_borders.empty:
    raise RuntimeError("shared_borders is missing or empty. Run Cell 1 first.")

# OSM query settings
BORDER_QUERY_BUFFER_DEG = 0.05  # about 5 km at equator
QUERY_BBOX_ROUND = 3  # bbox dedupe precision
MAX_BORDER_SEGMENTS = None  # set an int for quick tests, e.g. 20

OUTPUT_METHOD3 = Path.home() / "Desktop" / "border_point_method3_osmnx.gpkg"
LAYER_CONTROLS = "osm_border_controls"
LAYER_ROADS = "osm_border_roads"
LAYER_POINTS = "osm_border_crossing_points"

# Keep cache enabled for faster reruns
ox.settings.use_cache = True
ox.settings.log_console = False

# Tags: border controls / customs / checkpoints
control_tags = {
    "border_control": True,
    "barrier": "border_control",
    "customs": True,
    "amenity": "customs",
    "checkpoint": True,
    "immigration": True,
    "office": "government",
}

# Tags: roads near borders
road_tags = {
    "highway": [
        "motorway",
        "motorway_link",
        "trunk",
        "trunk_link",
        "primary",
        "primary_link",
        "secondary",
        "secondary_link",
        "tertiary",
        "tertiary_link",
    ]
}

def to_point_geometry(geom):
    if geom is None or geom.is_empty:
        return None
    if isinstance(geom, Point):
        return geom
    if isinstance(geom, MultiPoint):
        pts = [p for p in geom.geoms if p is not None and not p.is_empty]
        return pts[0] if pts else None
    return geom.representative_point()

def extract_points_local(geom):
    if geom is None or geom.is_empty:
        return []
    if isinstance(geom, Point):
        return [geom]
    if isinstance(geom, MultiPoint):
        return [p for p in geom.geoms if p is not None and not p.is_empty]
    if hasattr(geom, "geoms"):
        out = []
        for part in geom.geoms:
            out.extend(extract_points_local(part))
        return out
    return []

def ensure_gdf_4326(gdf):
    if gdf.crs is None:
        return gdf.set_crs(4326, allow_override=True)
    return gdf.to_crs(4326)

borders = shared_borders.copy()
borders = ensure_gdf_4326(borders)
borders = borders[borders.geometry.notnull() & ~borders.geometry.is_empty].copy()
if MAX_BORDER_SEGMENTS is not None:
    borders = borders.head(MAX_BORDER_SEGMENTS).copy()

print(f"[info] Border segments to query: {len(borders)}")

roads_chunks = []
controls_chunks = []
seen_bboxes = set()

for i, row in borders.iterrows():
    corridor = row.geometry.buffer(BORDER_QUERY_BUFFER_DEG)
    if corridor.is_empty:
        continue

    minx, miny, maxx, maxy = corridor.bounds
    bbox_key = (
        round(minx, QUERY_BBOX_ROUND),
        round(miny, QUERY_BBOX_ROUND),
        round(maxx, QUERY_BBOX_ROUND),
        round(maxy, QUERY_BBOX_ROUND),
    )
    if bbox_key in seen_bboxes:
        continue
    seen_bboxes.add(bbox_key)

    bbox = (minx, miny, maxx, maxy)

    try:
        roads_part = ox.features_from_bbox(bbox=bbox, tags=road_tags)
        if not roads_part.empty:
            roads_part = ensure_gdf_4326(gpd.GeoDataFrame(roads_part))
            roads_chunks.append(roads_part)
    except Exception as exc:
        print(f"[warn] roads query failed for bbox {bbox_key}: {exc}")

    try:
        controls_part = ox.features_from_bbox(bbox=bbox, tags=control_tags)
        if not controls_part.empty:
            controls_part = ensure_gdf_4326(gpd.GeoDataFrame(controls_part))
            controls_chunks.append(controls_part)
    except Exception as exc:
        print(f"[warn] controls query failed for bbox {bbox_key}: {exc}")

    if len(seen_bboxes) % 20 == 0:
        print(f"[info] Queried unique bboxes: {len(seen_bboxes)}")

print(f"[info] Total unique bboxes queried: {len(seen_bboxes)}")

# Build roads layer
if roads_chunks:
    roads = gpd.GeoDataFrame(pd.concat(roads_chunks, ignore_index=False), geometry="geometry", crs=4326)
    roads = roads.reset_index()
    roads = roads[roads.geometry.notnull() & ~roads.geometry.is_empty].copy()
    roads = roads[roads.geometry.geom_type.isin(["LineString", "MultiLineString"])].copy()
    dedup_cols = [c for c in ["element", "id", "osmid"] if c in roads.columns]
    if dedup_cols:
        roads = roads.drop_duplicates(subset=dedup_cols).copy()
    else:
        roads["_wkb"] = roads.geometry.apply(lambda g: g.wkb_hex if g is not None else None)
        roads = roads.drop_duplicates(subset=["_wkb"]).drop(columns=["_wkb"]).copy()
else:
    roads = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=4326)

print(f"[info] OSM border roads extracted: {len(roads)}")

# Build controls layer (point geometry)
if controls_chunks:
    controls = gpd.GeoDataFrame(pd.concat(controls_chunks, ignore_index=False), geometry="geometry", crs=4326)
    controls = controls.reset_index()
    controls = controls[controls.geometry.notnull() & ~controls.geometry.is_empty].copy()
    controls["geometry"] = controls.geometry.apply(to_point_geometry)
    controls = controls[controls.geometry.notnull() & ~controls.geometry.is_empty].copy()
    dedup_cols = [c for c in ["element", "id", "osmid"] if c in controls.columns]
    if dedup_cols:
        controls = controls.drop_duplicates(subset=dedup_cols).copy()
    else:
        controls["_wkb"] = controls.geometry.apply(lambda g: g.wkb_hex if g is not None else None)
        controls = controls.drop_duplicates(subset=["_wkb"]).drop(columns=["_wkb"]).copy()
else:
    controls = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=4326)

print(f"[info] OSM border controls extracted: {len(controls)}")

# Build crossing points from roads x shared borders
border_sindex = borders.sindex
cross_records = []

for _, road_row in roads.iterrows():
    road_geom = road_row.geometry
    if road_geom is None or road_geom.is_empty:
        continue

    candidate_idx = border_sindex.query(road_geom, predicate="intersects")
    if len(candidate_idx) == 0:
        continue

    road_attrs = road_row.drop(labels=["geometry"]).to_dict()
    for bidx in candidate_idx:
        b = borders.iloc[int(bidx)]
        inter = road_geom.intersection(b.geometry)
        pts = extract_points_local(inter)
        if not pts:
            continue

        for pt in pts:
            rec = dict(road_attrs)
            rec["country1"] = b.get("country1")
            rec["country2"] = b.get("country2")
            rec["iso3_1"] = b.get("iso3_1")
            rec["iso3_2"] = b.get("iso3_2")
            rec["lon"] = round(pt.x, 6)
            rec["lat"] = round(pt.y, 6)
            rec["geometry"] = pt
            cross_records.append(rec)

if cross_records:
    cross_points = gpd.GeoDataFrame(cross_records, geometry="geometry", crs=4326)
    dedup_cols = [c for c in ["iso3_1", "iso3_2", "lon", "lat"] if c in cross_points.columns]
    cross_points = cross_points.drop_duplicates(subset=dedup_cols).reset_index(drop=True)
else:
    cross_points = gpd.GeoDataFrame(columns=["geometry"], geometry="geometry", crs=4326)

print(f"[info] OSM crossing points extracted: {len(cross_points)}")

if OUTPUT_METHOD3.exists():
    OUTPUT_METHOD3.unlink()

controls.to_file(OUTPUT_METHOD3, layer=LAYER_CONTROLS, driver="GPKG")
roads.to_file(OUTPUT_METHOD3, layer=LAYER_ROADS, driver="GPKG", mode="a")
cross_points.to_file(OUTPUT_METHOD3, layer=LAYER_POINTS, driver="GPKG", mode="a")

print(f"[save] Method3 GeoPackage written: {OUTPUT_METHOD3}")
print(f"[save] Layers: {LAYER_CONTROLS}, {LAYER_ROADS}, {LAYER_POINTS}")
print(f"[save] Counts -> controls: {len(controls)}, roads: {len(roads)}, points: {len(cross_points)}")

cross_points.head()

[info] Border segments to query: 89


d:\Anaconda\envs\ox\Lib\site-packages\osmnx\_overpass.py:271: UserWarning: This area is 39 times your configured Overpass max query area size. It will automatically be divided up into multiple sub-queries accordingly. This may take a long time.
  multi_poly_proj = utils_geo._consolidate_subdivide_geometry(poly_proj)


In [ ]:
from pathlib import Path

# =========================
# Method 1 (comparison run)
# =========================

if "build_border_control_points_from_africa_edges" not in globals():
    raise RuntimeError("Method 1 function is not available. Run Cell 1 first.")
if "shared_borders" not in globals() or shared_borders is None or shared_borders.empty:
    raise RuntimeError("shared_borders is missing or empty. Run Cell 1 first.")
if "countries" not in globals() or countries is None or countries.empty:
    raise RuntimeError("countries is missing or empty. Run Cell 1 first.")

roads_gpkg_path = (
    AFRICA_ROADS_GPKG
    if "AFRICA_ROADS_GPKG" in globals()
    else Path(r"D:/geo data/ATSD road+rail+air/africa_roads_network.gpkg")
)
output_method1 = Path.home() / "Desktop" / "border_point_method1.gpkg"
points_layer_method1 = "border_control_points_method1"
borders_layer_method1 = "shared_borders_method1"

points_method1 = build_border_control_points_from_africa_edges(
    shared_borders=shared_borders,
    countries=countries,
    roads_gpkg_path=roads_gpkg_path,
)

if output_method1.exists():
    output_method1.unlink()

points_method1.to_file(output_method1, layer=points_layer_method1, driver="GPKG")
shared_borders.to_file(output_method1, layer=borders_layer_method1, driver="GPKG", mode="a")

print(f"[save] Method1 GeoPackage written: {output_method1}")
print(f"[save] Layers: {points_layer_method1}, {borders_layer_method1}")
print(f"[save] Method1 point count: {len(points_method1)}")

points_method1.head()